# LangGraph Basics — Graphs, State & Checkpoints (No LLM Calls)

**Raz Systems — Agentic AI Engineering**
*Prepared by Ajaz*

This notebook is a hands-on companion to the LangGraph slides. There are **no LLM calls anywhere** —
every "decision" is made with plain Python (`if`/`in` checks) so you can see exactly how LangGraph
moves data around, without an API key and without any randomness from a model.

We will build up in six small steps:

1. The simplest possible graph — one node
2. Chaining multiple nodes together
3. A **decision step** — routing a support message to **Billing** or **Technical** (the example from the article)
4. State **without** a checkpoint — what gets lost between runs
5. State **with** a checkpoint — what a checkpoint actually saves, step by step
6. Putting it together — the Billing/Technical router **with** memory across turns

Reference article: *How does LangGraph work?* — outcomeschool.com/blog/how-does-langgraph-work


In [20]:
# If langgraph isn't installed yet, uncomment the line below and run this cell.
# !pip install -q langgraph

import langgraph
print("LangGraph is ready to go.")


LangGraph is ready to go.


## 1. The Simplest Graph

Every LangGraph app needs three things:

- **State** — a shared box of data every node can read and write
- **Node(s)** — plain Python functions that read the state and return an update
- **Edges** — the wiring that says which node runs after which

Here is the smallest graph possible: `START -> greet -> END`.


In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


# 1. Define the State: what data flows through the graph
class GreetState(TypedDict):
    name: str
    message: str


# 2. Define a Node: a function that reads state and returns an update
def greet(state: GreetState):
    print(f"  [node: greet] read state -> name={state['name']!r}")
    update = {"message": f"Hello, {state['name']}! Welcome to LangGraph."}
    print(f"  [node: greet] returning update -> {update}")
    return update


# 3. Wire up the graph
builder = StateGraph(GreetState)
builder.add_node("greet", greet)
builder.add_edge(START, "greet")
builder.add_edge("greet", END)

graph = builder.compile()


In [3]:
# Run it
result = graph.invoke({"name": "Ajaz", "message": ""})
print("\nFinal state returned by invoke():")
print(result)


  [node: greet] read state -> name='Ajaz'
  [node: greet] returning update -> {'message': 'Hello, Ajaz! Welcome to LangGraph.'}

Final state returned by invoke():
{'name': 'Ajaz', 'message': 'Hello, Ajaz! Welcome to LangGraph.'}


**What just happened?**

- `invoke()` starts at `START`, LangGraph sees the edge to `greet`, runs it.
- The node only returned `{"message": ...}` — it did NOT need to return `name` again.
- LangGraph automatically **merges** whatever a node returns into the existing state.
- After `greet`, the edge to `END` finishes the run and the final state is returned.


## 2. Chaining Multiple Nodes

Now let's connect three nodes in a straight line: `START -> A -> B -> C -> END`,
matching the assembly-line idea from the slides. Each node adds one small piece to the state.


In [4]:
class OrderState(TypedDict):
    item: str
    steps: list


def take_order(state: OrderState):
    steps = state.get("steps", [])
    steps = steps + [f"Order received for: {state['item']}"]
    return {"steps": steps}


def pack_order(state: OrderState):
    steps = state["steps"] + ["Order packed"]
    return {"steps": steps}


def ship_order(state: OrderState):
    steps = state["steps"] + ["Order shipped"]
    return {"steps": steps}


builder = StateGraph(OrderState)
builder.add_node("take_order", take_order)
builder.add_node("pack_order", pack_order)
builder.add_node("ship_order", ship_order)

builder.add_edge(START, "take_order")
builder.add_edge("take_order", "pack_order")
builder.add_edge("pack_order", "ship_order")
builder.add_edge("ship_order", END)

order_graph = builder.compile()


In [5]:
result = order_graph.invoke({"item": "Wireless Mouse", "steps": []})
for i, step in enumerate(result["steps"], start=1):
    print(f"{i}. {step}")


1. Order received for: Wireless Mouse
2. Order packed
3. Order shipped


Each node only knew about **its own job**. `pack_order` didn't need to know how the order was
taken, and `ship_order` didn't need to know how it was packed — they just read the shared `steps`
list from state and added to it. This is the "assembly line" idea in code.


## 3. A Decision Step: Billing vs Technical Support

This is the example from the article. A plain chain can't easily make a decision and go down
a different path — but a **conditional edge** can. It's a fork in the road: LangGraph runs a
router function, looks at what it returns, and sends the graph to the matching node.


In [6]:
class SupportState(TypedDict):
    message: str
    reply: str


def route_message(state: SupportState):
    # This is the "fork in the road". No LLM here -- just a simple keyword check.
    text = state["message"].lower()
    if "bill" in text or "payment" in text or "invoice" in text:
        decision = "billing"
    else:
        decision = "technical"
    print(f"  [router] message={state['message']!r} -> decision={decision!r}")
    return decision


def billing(state: SupportState):
    return {"reply": "This is the Billing team. We'll help you with your payment."}


def technical(state: SupportState):
    return {"reply": "This is Technical Support. Let's troubleshoot the issue."}


builder = StateGraph(SupportState)
builder.add_node("billing", billing)
builder.add_node("technical", technical)

# add_conditional_edges(source, router_function, mapping)
# The mapping tells LangGraph: if router returns "billing", go to the "billing" node, etc.
builder.add_conditional_edges(
    START,
    route_message,
    {"billing": "billing", "technical": "technical"},
)
builder.add_edge("billing", END)
builder.add_edge("technical", END)

support_graph = builder.compile()


In [7]:
test_messages = [
    "I have a question about my last invoice",
    "My laptop won't connect to WiFi",
    "Can you help me update my payment method?",
    "The app keeps crashing on startup",
]

for msg in test_messages:
    result = support_graph.invoke({"message": msg, "reply": ""})
    print(f"User: {msg}\n  -> {result['reply']}\n")


  [router] message='I have a question about my last invoice' -> decision='billing'
User: I have a question about my last invoice
  -> This is the Billing team. We'll help you with your payment.

  [router] message="My laptop won't connect to WiFi" -> decision='technical'
User: My laptop won't connect to WiFi
  -> This is Technical Support. Let's troubleshoot the issue.

  [router] message='Can you help me update my payment method?' -> decision='billing'
User: Can you help me update my payment method?
  -> This is the Billing team. We'll help you with your payment.

  [router] message='The app keeps crashing on startup' -> decision='technical'
User: The app keeps crashing on startup
  -> This is Technical Support. Let's troubleshoot the issue.



Notice the graph itself never contains an `if/else` about billing vs. technical — that logic
lives inside `route_message`. The graph just says: *"run the router, then go wherever it points."*
This is exactly the pattern the slides call **Conditional Edges: A Fork in the Road**.


## 4. State WITHOUT a Checkpoint

By default, a compiled graph has **no memory** between separate `invoke()` calls. Every call
starts from exactly the input you gave it — nothing from a previous call carries over.

Let's build a tiny "ticket counter" graph and call it three times in a row.


In [21]:
class TicketState(TypedDict):
    customer: str
    tickets_handled: int


def handle_ticket(state: TicketState):
    current = state.get("tickets_handled", 0)
    new_total = current + 1
    print(f"  [handle_ticket] saw tickets_handled={current} -> writing {new_total}")
    return {"tickets_handled": new_total}


builder = StateGraph(TicketState)
builder.add_node("handle_ticket", handle_ticket)
builder.add_edge(START, "handle_ticket")
builder.add_edge("handle_ticket", END)

no_memory_graph = builder.compile()  # <-- no checkpointer passed in


In [22]:
print("Calling invoke() three times for the same customer, WITHOUT a checkpoint:\n")
for call_number in range(1, 4):
    result = no_memory_graph.invoke({"customer": "Ajaz", "tickets_handled": 0})
    print(f"Call #{call_number} result: {result}\n")


Calling invoke() three times for the same customer, WITHOUT a checkpoint:

  [handle_ticket] saw tickets_handled=0 -> writing 1
Call #1 result: {'customer': 'Ajaz', 'tickets_handled': 1}

  [handle_ticket] saw tickets_handled=0 -> writing 1
Call #2 result: {'customer': 'Ajaz', 'tickets_handled': 1}

  [handle_ticket] saw tickets_handled=0 -> writing 1
Call #3 result: {'customer': 'Ajaz', 'tickets_handled': 1}



**`tickets_handled` is `1` every single time.**

Nothing was remembered between calls — each `invoke()` is a fresh, isolated run. If we hadn't
explicitly passed `"tickets_handled": 0` as input, the graph wouldn't know the customer had ever
been helped before. This is the exact gap that a **checkpointer** closes.


## 5. State WITH a Checkpoint

A **checkpointer** saves the state after every step of a run, keyed by a `thread_id`. The next
time you `invoke()` with the *same* `thread_id`, LangGraph loads the saved state first — so the
graph picks up exactly where it left off.

We'll use `MemorySaver`, an in-memory checkpointer that's perfect for demos (in production you'd
typically use a database-backed checkpointer instead).


In [23]:
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()

# Recompile the SAME graph definition, this time with a checkpointer attached
builder = StateGraph(TicketState)
builder.add_node("handle_ticket", handle_ticket)
builder.add_edge(START, "handle_ticket")
builder.add_edge("handle_ticket", END)

memory_graph = builder.compile(checkpointer=checkpointer)


In [24]:
# A config with a thread_id tells LangGraph WHICH saved conversation/session to use
config_ajaz = {"configurable": {"thread_id": "customer-ajaz"}}

print("Calling invoke() three times for the same customer, WITH a checkpoint:\n")
for call_number in range(1, 4):
    # Notice: we only pass "customer" the first time in spirit — the checkpoint
    # will supply tickets_handled from the previous run automatically.
    result = memory_graph.invoke({"customer": "ajaz"}, config=config_ajaz)
    print(f"Call #{call_number} result: {result}\n")


Calling invoke() three times for the same customer, WITH a checkpoint:

  [handle_ticket] saw tickets_handled=0 -> writing 1
Call #1 result: {'customer': 'ajaz', 'tickets_handled': 1}

  [handle_ticket] saw tickets_handled=1 -> writing 2
Call #2 result: {'customer': 'ajaz', 'tickets_handled': 2}

  [handle_ticket] saw tickets_handled=2 -> writing 3
Call #3 result: {'customer': 'ajaz', 'tickets_handled': 3}



**`tickets_handled` now climbs: 1, 2, 3.**

Because every call used the same `thread_id` (`"customer-amit"`), LangGraph:

1. Loaded the last saved state for that thread before running,
2. Ran `handle_ticket` on top of it,
3. Saved the new state back under the same `thread_id`.

Let's prove the checkpoint is doing this — and not just Python variables in memory — by
inspecting the saved state directly with `get_state()`.


In [26]:
snapshot = memory_graph.get_state(config_ajaz)
print("Saved state for thread 'customer-ajaz':")
print(" ", snapshot.values)
print("\nCheckpoint metadata (which step this was, etc.):")
print(" ", snapshot.metadata)


Saved state for thread 'customer-ajaz':
  {'customer': 'ajaz', 'tickets_handled': 3}

Checkpoint metadata (which step this was, etc.):
  {'source': 'loop', 'step': 7, 'parents': {}}


Now let's use a **different** `thread_id` for a different customer, and prove the two
threads never mix.


In [27]:
config_aiza = {"configurable": {"thread_id": "customer-aiza"}}

print("A brand-new thread_id starts completely fresh:\n")
result = memory_graph.invoke({"customer": "Aiza"}, config=config_aiza)
print("Aiza's first call:", result)

result = memory_graph.invoke({"customer": "Aiza"}, config=config_aiza)
print("Aiza's second call:", result)

print("\nMeanwhile Amit's thread is untouched — check it again:")
print(" ", memory_graph.get_state(config_ajaz).values)


A brand-new thread_id starts completely fresh:

  [handle_ticket] saw tickets_handled=0 -> writing 1
Aiza's first call: {'customer': 'Aiza', 'tickets_handled': 1}
  [handle_ticket] saw tickets_handled=1 -> writing 2
Aiza's second call: {'customer': 'Aiza', 'tickets_handled': 2}

Meanwhile Amit's thread is untouched — check it again:
  {'customer': 'ajaz', 'tickets_handled': 3}


**Side-by-side summary**

| | Without checkpoint | With checkpoint (`MemorySaver`) |
|---|---|---|
| Each `invoke()` | Starts from scratch | Loads the last saved state for that `thread_id` |
| `tickets_handled` across 3 calls | `1, 1, 1` | `1, 2, 3` |
| Different customers | N/A — no memory to mix up | Kept separate by `thread_id` |
| Can you inspect saved state? | Nothing to inspect | Yes — `graph.get_state(config)` |


## 6. Putting It Together — Billing/Technical Router *with* Memory

Now let's combine Section 3 (the decision step) with Section 5 (checkpointing). Every message
still gets routed to Billing or Technical using the same simple keyword check — but now we'll
also track **how many tickets each customer has had handled**, persisted across turns.


In [28]:
class SupportMemoryState(TypedDict):
    customer: str
    message: str
    reply: str
    tickets_handled: int


def route_message_2(state: SupportMemoryState):
    text = state["message"].lower()
    if "bill" in text or "payment" in text or "invoice" in text:
        return "billing"
    return "technical"


def billing_2(state: SupportMemoryState):
    total = state.get("tickets_handled", 0) + 1
    return {
        "reply": "Billing team here — we'll sort out your payment question.",
        "tickets_handled": total,
    }


def technical_2(state: SupportMemoryState):
    total = state.get("tickets_handled", 0) + 1
    return {
        "reply": "Technical Support here — let's fix that for you.",
        "tickets_handled": total,
    }


builder = StateGraph(SupportMemoryState)
builder.add_node("billing", billing_2)
builder.add_node("technical", technical_2)
builder.add_conditional_edges(
    START, route_message_2, {"billing": "billing", "technical": "technical"}
)
builder.add_edge("billing", END)
builder.add_edge("technical", END)

support_with_memory = builder.compile(checkpointer=MemorySaver())


In [29]:
config_customer = {"configurable": {"thread_id": "customer-raz-001"}}

conversation = [
    "My WiFi keeps dropping every few minutes",
    "Also, can you check my last invoice?",
    "The app crashed again after that",
]

for turn, msg in enumerate(conversation, start=1):
    result = support_with_memory.invoke(
        {"customer": "Raz Systems Student", "message": msg}, config=config_customer
    )
    print(f"Turn {turn}: {msg}")
    print(f"  -> routed reply: {result['reply']}")
    print(f"  -> tickets_handled so far (from checkpoint): {result['tickets_handled']}\n")


Turn 1: My WiFi keeps dropping every few minutes
  -> routed reply: Technical Support here — let's fix that for you.
  -> tickets_handled so far (from checkpoint): 1

Turn 2: Also, can you check my last invoice?
  -> routed reply: Billing team here — we'll sort out your payment question.
  -> tickets_handled so far (from checkpoint): 2

Turn 3: The app crashed again after that
  -> routed reply: Technical Support here — let's fix that for you.
  -> tickets_handled so far (from checkpoint): 3



Each turn was routed independently by `route_message_2` (billing, then billing again,
then technical) — the **routing logic never needed memory**. But `tickets_handled` kept climbing
across turns (1, 2, 3) because the **checkpoint** carried the running total forward, regardless
of which node handled the ticket.

This is the same principle from the slides:
> *State is what LangGraph checkpoints to disk so a graph can pause, resume, and remember across
> many runs — a more robust foundation than manually re-sending chat history.*


## Recap

| Concept | What we saw in this notebook |
|---|---|
| **State** | A `TypedDict` box of data (`GreetState`, `OrderState`, `SupportState`, ...) |
| **Node** | A plain function reading state, returning a partial update |
| **Edge** | `add_edge(a, b)` — always go from `a` straight to `b` |
| **Conditional Edge** | `add_conditional_edges(...)` — a router function picks the next node |
| **No checkpoint** | Every `invoke()` is isolated; nothing carries over |
| **Checkpoint (`MemorySaver`)** | State is saved per `thread_id`; `invoke()` resumes from the last save |



None of this required an LLM — every decision was a plain `if`/`in` check in Python. In a real
app, you'd swap `route_message` for an LLM call that decides the category, and swap `billing`/
`technical` for nodes that actually call an LLM to draft a reply — but the **graph, state, and
checkpoint mechanics stay exactly the same.**

*Raz Systems — Agentic AI Engineering*
